# 6b Compare Reviews Rephrased

This notebook runs exact-n matched review-diversity analyses on the rephrased-text review branch.

## Analysis Sections

1. Configuration and exact-n panel setup
2. Whole-review paired diversity analyses
3. Field-specific strengths and weakness analyses
4. Facet Diversity (M1-M5): paired review panels plus exploratory pooled displacement
5. Cross-condition exports for synthesis

The facet block follows `docs/plans/add_diversity_metrics_spec_jul14.md`: review facets are computed per target proposal using exact-n panels and aggregated with paired Wilcoxon tests. Strengths and weakness are secondary rephrased-branch checks.


In [5]:
CONDITIONS_TO_RUN = ['baseline', 'one_at_a_time', 'persona']
TEXT_VERSION = 'rephrased'
COMPARISONS = ['all_ai', 'claude', 'gemini', 'gpt']
FIELD_BRANCHES = ['whole_review', 'strengths', 'weakness']
CONFIRMATORY_METRICS = ['mean_pairwise', 'nn']
EXPLORATORY_METRICS = [
    'centroid_loo',
    'global_centroid_dist',
    'medoid_dist',
    'span90',
    'mst_dispersion',
    'sparseness',
]
COMPATIBILITY_METRICS = ['remote_clique']
ALL_METRICS = CONFIRMATORY_METRICS + EXPLORATORY_METRICS + COMPATIBILITY_METRICS
MIN_HUMAN_PANEL_FOR_SENSITIVITY = 3
REUSE_ORIGINAL_BRANCH_COMBINATIONS = True


In [6]:
import itertools
import json
import os
from dataclasses import replace
from pathlib import Path

os.environ.setdefault('MPLCONFIGDIR', str(Path('/tmp') / 'codex-matplotlib'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import wilcoxon
from sklearn.decomposition import PCA

import sys
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from diversity_inference import (
    build_review_facet_outputs,
    write_facet_outputs,
)
from compare_review_diversity import (
    COMPARISON_LABELS,
    ReviewConditionAnalysis,
    build_exact_n_panel_combinations,
    build_review_diversity_proposal_master,
    compute_review_metric_correlation_table,
    cross_condition_summary_table,
    load_pickle,
    load_review_analysis_inputs,
    paired_review_diversity_tests,
    plot_review_diversity_effects,
    plot_review_diversity_paired_slopes,
    plot_review_embedding_space,
    save_pickle,
)
from proposal_generation import find_project_root

PROJECT_ROOT = find_project_root(PROJECT_ROOT)
sns.set_theme(style='whitegrid', context='talk')

FIELD_FILE_MAP = {
    'whole_review': {
        'embedding': 'review_embeddings_text.pkl',
        'pairwise': 'review_pairwise_cosine_text.npy',
    },
    'strengths': {
        'embedding': 'review_embeddings_strengths.pkl',
        'pairwise': 'review_pairwise_cosine_strengths.npy',
    },
    'weakness': {
        'embedding': 'review_embeddings_weakness.pkl',
        'pairwise': 'review_pairwise_cosine_weakness.npy',
    },
}

def cross_condition_ai_contrast_table(proposal_master_df):
    rows = []
    conditions = sorted(proposal_master_df['condition'].dropna().unique().tolist())
    for comparison in sorted(proposal_master_df['comparison'].dropna().unique().tolist()):
        sub = proposal_master_df[proposal_master_df['comparison'] == comparison].copy()
        for metric in sorted(sub['metric'].dropna().unique().tolist()):
            metric_df = sub[sub['metric'] == metric][['condition', 'target_proposal_uid', 'ai_metric_panel_mean']].dropna()
            for left, right in itertools.combinations(conditions, 2):
                left_df = metric_df[metric_df['condition'] == left][['target_proposal_uid', 'ai_metric_panel_mean']].rename(columns={'ai_metric_panel_mean': 'left_value'})
                right_df = metric_df[metric_df['condition'] == right][['target_proposal_uid', 'ai_metric_panel_mean']].rename(columns={'ai_metric_panel_mean': 'right_value'})
                merged = left_df.merge(right_df, on='target_proposal_uid', how='inner')
                if merged.empty:
                    continue
                diff = merged['left_value'] - merged['right_value']
                if np.allclose(diff, 0):
                    stat = 0.0
                    p_value = 1.0
                else:
                    test = wilcoxon(merged['left_value'], merged['right_value'], zero_method='wilcox', alternative='two-sided', mode='auto')
                    stat = float(test.statistic)
                    p_value = float(test.pvalue)
                rows.append({
                    'comparison': comparison,
                    'comparison_label': COMPARISON_LABELS.get(comparison, comparison),
                    'metric': metric,
                    'left_condition': left,
                    'right_condition': right,
                    'n_proposals': int(len(merged)),
                    'left_mean_ai_metric': float(merged['left_value'].mean()),
                    'right_mean_ai_metric': float(merged['right_value'].mean()),
                    'mean_difference_left_minus_right': float(diff.mean()),
                    'median_difference_left_minus_right': float(diff.median()),
                    'wilcoxon_statistic': stat,
                    'p_value': p_value,
                })
    return pd.DataFrame(rows)

def load_field_analysis(project_root, condition, field_branch):
    if field_branch == 'whole_review':
        return load_review_analysis_inputs(project_root, condition, text_version='rephrased')
    base_analysis = load_review_analysis_inputs(project_root, condition, text_version='rephrased')
    branch_root = project_root / 'data' / 'prepared' / condition / 'reviews' / 'rephrased'
    embedding_path = branch_root / FIELD_FILE_MAP[field_branch]['embedding']
    pairwise_path = branch_root / FIELD_FILE_MAP[field_branch]['pairwise']
    if not embedding_path.exists() or not pairwise_path.exists():
        raise FileNotFoundError(f'Missing field-specific prepared assets for {condition}/{field_branch}: {embedding_path}, {pairwise_path}')
    field_embeddings = load_pickle(embedding_path)
    field_pairwise = np.load(pairwise_path)
    master_uids = base_analysis.review_master['review_uid'].astype(str).tolist()
    field_uids = [str(uid) for uid in field_embeddings.get('review_uids', [])]
    if master_uids != field_uids:
        raise RuntimeError(f'{condition}/{field_branch}: field embedding review_uid roster does not match rephrased review master')
    if field_pairwise.shape != (len(master_uids), len(master_uids)):
        raise RuntimeError(f'{condition}/{field_branch}: field pairwise matrix shape mismatch: {field_pairwise.shape}')
    return replace(base_analysis, review_embeddings=field_embeddings, review_pairwise=field_pairwise)

def load_or_build_rephrased_combinations(project_root, condition, rephrased_analysis, rephrased_shared_cache_dir):
    local_cache_path = rephrased_shared_cache_dir / 'exact_n_panel_combinations.pkl'
    original_cache_path = project_root / 'results' / 'tables' / condition / 'reviews' / 'original' / 'shared_cache' / 'exact_n_panel_combinations.pkl'
    reuse_original = False
    combination_cache = None

    if REUSE_ORIGINAL_BRANCH_COMBINATIONS and original_cache_path.exists():
        original_analysis = load_review_analysis_inputs(project_root, condition, text_version='original')
        same_panel_registry = rephrased_analysis.panel_registry.equals(original_analysis.panel_registry)
        same_sampling_frame = rephrased_analysis.sampling_frame.equals(original_analysis.sampling_frame)
        same_uid_roster = rephrased_analysis.review_master['review_uid'].astype(str).tolist() == original_analysis.review_master['review_uid'].astype(str).tolist()
        if same_panel_registry and same_sampling_frame and same_uid_roster:
            combination_cache = load_pickle(original_cache_path)
            reuse_original = True

    if combination_cache is None:
        combination_cache = {comparison: build_exact_n_panel_combinations(rephrased_analysis.panel_registry, comparison) for comparison in COMPARISONS}

    save_pickle(local_cache_path, combination_cache)
    metadata = {
        'condition': condition,
        'text_version': 'rephrased',
        'reused_original_branch_cache': reuse_original,
        'original_cache_path': str(original_cache_path) if original_cache_path.exists() else '',
        'local_cache_path': str(local_cache_path),
    }
    (rephrased_shared_cache_dir / 'exact_n_panel_combinations_metadata.json').write_text(json.dumps(metadata, indent=2))
    return combination_cache, metadata

def plot_field_embedding_pca(analysis, output_path, title):
    embeddings = np.asarray(analysis.review_embeddings['embeddings'], dtype=float)
    if embeddings.ndim != 2 or embeddings.shape[0] != len(analysis.review_master):
        raise RuntimeError('Embedding array does not align with review_master for PCA diagnostic')
    coords = PCA(n_components=2, random_state=42).fit_transform(embeddings)
    plot_df = analysis.review_master[['review_source', 'source_family', 'target_cohort']].copy()
    plot_df['x'] = coords[:, 0]
    plot_df['y'] = coords[:, 1]
    plot_df['group_label'] = np.where(plot_df['review_source'] == 'human', 'Human', plot_df['source_family'].str.title())
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.scatterplot(data=plot_df, x='x', y='y', hue='group_label', style='target_cohort', alpha=0.7, s=40, ax=ax)
    ax.set_title(title)
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    fig.tight_layout()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=200, bbox_inches='tight')
    plt.close(fig)

def export_field_outputs(field_branch, proposal_master_df, panel_long_df, tests_df, metric_corr_df, tables_all_ai_dir, tables_per_model_dir, shared_cache_dir):
    shared_name = 'review_diversity_proposal_master.csv' if field_branch == 'whole_review' else f'review_diversity_{field_branch}_proposal_master.csv'
    panels_name = 'review_diversity_panels_long.csv' if field_branch == 'whole_review' else f'review_diversity_{field_branch}_panels_long.csv'
    corr_name = 'review_diversity_metric_correlations.csv' if field_branch == 'whole_review' else f'review_diversity_{field_branch}_metric_correlations.csv'
    proposal_master_df.to_csv(shared_cache_dir / shared_name, index=False)
    panel_long_df.to_csv(shared_cache_dir / panels_name, index=False)
    metric_corr_df.to_csv(shared_cache_dir / corr_name, index=False)

    all_ai_master_df = proposal_master_df[proposal_master_df['comparison'] == 'all_ai'].copy()
    per_model_master_df = proposal_master_df[proposal_master_df['comparison'].isin(['claude', 'gemini', 'gpt'])].copy()
    all_ai_panels_df = panel_long_df[panel_long_df['comparison'] == 'all_ai'].copy()
    per_model_panels_df = panel_long_df[panel_long_df['comparison'].isin(['claude', 'gemini', 'gpt'])].copy()

    if field_branch == 'whole_review':
        all_ai_master_df.to_csv(tables_all_ai_dir / 'review_diversity_proposal_master.csv', index=False)
        per_model_master_df.to_csv(tables_per_model_dir / 'review_diversity_proposal_master.csv', index=False)
        all_ai_panels_df.to_csv(tables_all_ai_dir / 'review_diversity_panels_long.csv', index=False)
        per_model_panels_df.to_csv(tables_per_model_dir / 'review_diversity_panels_long.csv', index=False)
        tests_df[tests_df['comparison'] == 'all_ai'].to_csv(tables_all_ai_dir / 'review_diversity_tests_human_vs_allai.csv', index=False)
        tests_df[tests_df['comparison'].isin(['claude', 'gemini', 'gpt'])].to_csv(tables_per_model_dir / 'review_diversity_tests_human_vs_model.csv', index=False)
    else:
        suffix = field_branch
        all_ai_master_df.to_csv(tables_all_ai_dir / f'review_diversity_{suffix}_proposal_master.csv', index=False)
        per_model_master_df.to_csv(tables_per_model_dir / f'review_diversity_{suffix}_proposal_master.csv', index=False)
        all_ai_panels_df.to_csv(tables_all_ai_dir / f'review_diversity_{suffix}_panels_long.csv', index=False)
        per_model_panels_df.to_csv(tables_per_model_dir / f'review_diversity_{suffix}_panels_long.csv', index=False)
        tests_df.to_csv(shared_cache_dir / f'review_diversity_{suffix}_tests.csv', index=False)

print(f'Project root: {PROJECT_ROOT}')
print(f'Conditions: {CONDITIONS_TO_RUN}')


Project root: /Users/eveyhuang/Documents/NICO/human-AI-proposal
Conditions: ['baseline', 'one_at_a_time', 'persona']


In [7]:
condition_outputs = {}
cross_condition_whole_test_frames = []
cross_condition_whole_proposal_frames = []
cross_condition_field_test_frames = []
cross_condition_facet_test_frames = []
cross_condition_facet_gradient_frames = []
cross_condition_facet_curve_frames = []
cross_condition_facet_paired_frames = []

for condition in CONDITIONS_TO_RUN:
    print(f'\n=== 6b rephrased review comparison: {condition} ===')
    rephrased_analysis = load_review_analysis_inputs(PROJECT_ROOT, condition, text_version=TEXT_VERSION)
    if rephrased_analysis.panel_registry['target_proposal_uid'].nunique() != 23:
        raise RuntimeError(f'{condition}: expected 23 target proposals, found {rephrased_analysis.panel_registry["target_proposal_uid"].nunique()}')

    tables_all_ai_dir = PROJECT_ROOT / 'results' / 'tables' / condition / 'reviews' / TEXT_VERSION / 'all_ai'
    figures_all_ai_dir = PROJECT_ROOT / 'results' / 'figures' / condition / 'reviews' / TEXT_VERSION / 'all_ai'
    tables_per_model_dir = PROJECT_ROOT / 'results' / 'tables' / condition / 'reviews' / TEXT_VERSION / 'per_model'
    figures_per_model_dir = PROJECT_ROOT / 'results' / 'figures' / condition / 'reviews' / TEXT_VERSION / 'per_model'
    shared_cache_dir = PROJECT_ROOT / 'results' / 'tables' / condition / 'reviews' / TEXT_VERSION / 'shared_cache'
    facet_tables_dir = PROJECT_ROOT / 'results' / 'tables' / condition / 'reviews' / TEXT_VERSION / 'facet'
    facet_figures_dir = PROJECT_ROOT / 'results' / 'figures' / condition / 'reviews' / TEXT_VERSION / 'facet'
    for path in [tables_all_ai_dir, figures_all_ai_dir, tables_per_model_dir, figures_per_model_dir, shared_cache_dir, facet_tables_dir, facet_figures_dir]:
        path.mkdir(parents=True, exist_ok=True)

    combination_cache, combination_meta = load_or_build_rephrased_combinations(PROJECT_ROOT, condition, rephrased_analysis, shared_cache_dir)
    for comparison in COMPARISONS:
        eligible_targets = [target_uid for target_uid, payload in combination_cache[comparison].items() if payload.get('eligible')]
        if len(eligible_targets) != 23:
            raise RuntimeError(f'{condition}/{comparison}: expected 23 eligible proposals, found {len(eligible_targets)}')

    branch_results = {}
    condition_facet_tests = []
    condition_facet_gradients = []
    condition_facet_curves = []
    condition_facet_paired = []
    for field_branch in FIELD_BRANCHES:
        analysis = load_field_analysis(PROJECT_ROOT, condition, field_branch)
        proposal_master_df, panel_long_df = build_review_diversity_proposal_master(analysis, combination_cache)
        if proposal_master_df.empty or panel_long_df.empty:
            raise RuntimeError(f'{condition}/{field_branch}: review-diversity outputs are empty')
        proposal_master_df = proposal_master_df.copy()
        panel_long_df = panel_long_df.copy()
        proposal_master_df['field_branch'] = field_branch
        panel_long_df['field_branch'] = field_branch

        all_tests_df = paired_review_diversity_tests(proposal_master_df, subset_label='all_proposals')
        restricted_df = proposal_master_df[proposal_master_df['target_human_n_reviews'] >= MIN_HUMAN_PANEL_FOR_SENSITIVITY].copy()
        restricted_tests_df = paired_review_diversity_tests(restricted_df, subset_label=f'human_n_gte_{MIN_HUMAN_PANEL_FOR_SENSITIVITY}')
        tests_df = pd.concat([all_tests_df, restricted_tests_df], ignore_index=True)
        tests_df.insert(0, 'condition', condition)
        tests_df['field_branch'] = field_branch

        metric_corr_df = compute_review_metric_correlation_table(proposal_master_df)
        metric_corr_df.insert(0, 'condition', condition)
        metric_corr_df['field_branch'] = field_branch

        export_field_outputs(field_branch, proposal_master_df, panel_long_df, tests_df, metric_corr_df, tables_all_ai_dir, tables_per_model_dir, shared_cache_dir)


        facet_field = 'whole' if field_branch == 'whole_review' else field_branch
        facet_tests_df, facet_gradient_df, facet_curves_df, facet_paired_df = build_review_facet_outputs(
            analysis,
            combination_cache,
            text_branch=TEXT_VERSION,
            field=facet_field,
            n_boot=1000,
            seed=42,
        )
        condition_facet_tests.append(facet_tests_df)
        condition_facet_gradients.append(facet_gradient_df)
        condition_facet_curves.append(facet_curves_df)
        condition_facet_paired.append(facet_paired_df)

        branch_results[field_branch] = {
            'analysis': analysis,
            'proposal_master_df': proposal_master_df,
            'panel_long_df': panel_long_df,
            'tests_df': tests_df,
            'metric_corr_df': metric_corr_df,
            'facet_tests_df': facet_tests_df,
            'facet_gradient_df': facet_gradient_df,
            'facet_paired_df': facet_paired_df,
        }

        if field_branch == 'whole_review':
            cross_condition_whole_test_frames.append(tests_df)
            cross_condition_whole_proposal_frames.append(proposal_master_df)
        else:
            cross_condition_field_test_frames.append(tests_df)


    condition_facet_tests_df = pd.concat(condition_facet_tests, ignore_index=True) if condition_facet_tests else pd.DataFrame()
    condition_facet_gradient_df = pd.concat(condition_facet_gradients, ignore_index=True) if condition_facet_gradients else pd.DataFrame()
    condition_facet_curves_df = pd.concat(condition_facet_curves, ignore_index=True) if condition_facet_curves else pd.DataFrame()
    condition_facet_paired_df = pd.concat(condition_facet_paired, ignore_index=True) if condition_facet_paired else pd.DataFrame()
    write_facet_outputs(
        facet_tables_dir,
        facet_figures_dir,
        condition_facet_tests_df,
        condition_facet_gradient_df,
        condition_facet_curves_df,
        condition_facet_paired_df,
    )
    cross_condition_facet_test_frames.append(condition_facet_tests_df)
    cross_condition_facet_gradient_frames.append(condition_facet_gradient_df)
    cross_condition_facet_curve_frames.append(condition_facet_curves_df)
    cross_condition_facet_paired_frames.append(condition_facet_paired_df)

    plot_review_diversity_paired_slopes(
        branch_results['whole_review']['proposal_master_df'],
        comparison='all_ai',
        metrics=CONFIRMATORY_METRICS,
        output_path=figures_all_ai_dir / 'paired_review_diversity_confirmatory.png',
        title=f'{condition}: Human vs All AI exact-n matched review diversity (rephrased)',
    )
    plot_review_diversity_effects(
        branch_results['whole_review']['tests_df'][
            (branch_results['whole_review']['tests_df']['comparison'] == 'all_ai')
            & (branch_results['whole_review']['tests_df']['subset_label'] == 'all_proposals')
        ],
        output_path=figures_all_ai_dir / 'review_diversity_effects_confirmatory.png',
        title=f'{condition}: Human vs All AI review-diversity effects (rephrased)',
        metric_class='confirmatory',
    )
    plot_review_diversity_effects(
        branch_results['whole_review']['tests_df'][
            (branch_results['whole_review']['tests_df']['comparison'].isin(['claude', 'gemini', 'gpt']))
            & (branch_results['whole_review']['tests_df']['subset_label'] == 'all_proposals')
        ],
        output_path=figures_per_model_dir / 'review_diversity_effects_confirmatory.png',
        title=f'{condition}: Human vs model review-diversity effects (rephrased)',
        metric_class='confirmatory',
    )
    plot_review_embedding_space(
        branch_results['whole_review']['analysis'],
        output_path=figures_all_ai_dir / 'review_space_umap.png',
        title=f'{condition}: review-space UMAP (rephrased whole review)',
    )

    strengths_all = branch_results['strengths']['tests_df'][(branch_results['strengths']['tests_df']['comparison'] == 'all_ai') & (branch_results['strengths']['tests_df']['subset_label'] == 'all_proposals') & (branch_results['strengths']['tests_df']['metric_class'] == 'confirmatory')].copy()
    weakness_all = branch_results['weakness']['tests_df'][(branch_results['weakness']['tests_df']['comparison'] == 'all_ai') & (branch_results['weakness']['tests_df']['subset_label'] == 'all_proposals') & (branch_results['weakness']['tests_df']['metric_class'] == 'confirmatory')].copy()
    strengths_all['field_label'] = 'Strengths'
    weakness_all['field_label'] = 'Weakness'
    field_effect_df = pd.concat([strengths_all, weakness_all], ignore_index=True)
    fig, axes = plt.subplots(1, len(CONFIRMATORY_METRICS), figsize=(6 * len(CONFIRMATORY_METRICS), 5), sharey=False)
    if len(CONFIRMATORY_METRICS) == 1:
        axes = [axes]
    for ax, metric in zip(axes, CONFIRMATORY_METRICS):
        sub = field_effect_df[field_effect_df['metric'] == metric].copy()
        sns.barplot(data=sub, x='field_label', y='effect_human_minus_ai_mean', ax=ax)
        ax.axhline(0.0, color='black', linewidth=1.0)
        ax.set_title(metric)
        ax.set_xlabel('')
        ax.set_ylabel('Mean paired difference (Human - AI)')
    fig.suptitle(f'{condition}: strengths vs weakness review-diversity effects')
    fig.tight_layout()
    fig.savefig(figures_all_ai_dir / 'strengths_vs_weakness_effects_confirmatory.png', dpi=200, bbox_inches='tight')
    plt.close(fig)

    plot_field_embedding_pca(
        branch_results['strengths']['analysis'],
        output_path=figures_all_ai_dir / 'strengths_embedding_pca.png',
        title=f'{condition}: strengths embedding diagnostic (rephrased)',
    )
    plot_field_embedding_pca(
        branch_results['weakness']['analysis'],
        output_path=figures_all_ai_dir / 'weakness_embedding_pca.png',
        title=f'{condition}: weakness embedding diagnostic (rephrased)',
    )

    condition_outputs[condition] = {
        'combination_meta': combination_meta,
        'branch_results': branch_results,
        'tables_all_ai_dir': tables_all_ai_dir,
        'tables_per_model_dir': tables_per_model_dir,
        'figures_all_ai_dir': figures_all_ai_dir,
        'figures_per_model_dir': figures_per_model_dir,
        'shared_cache_dir': shared_cache_dir,
        'facet_tests_df': condition_facet_tests_df,
        'facet_gradient_df': condition_facet_gradient_df,
    }

    print('Saved condition outputs:')
    print(f'  reused original panel cache: {combination_meta["reused_original_branch_cache"]}')
    print(f'  whole-review master: {shared_cache_dir / "review_diversity_proposal_master.csv"}')
    print(f'  strengths master: {shared_cache_dir / "review_diversity_strengths_proposal_master.csv"}')
    print(f'  weakness master: {shared_cache_dir / "review_diversity_weakness_proposal_master.csv"}')



=== 6b rephrased review comparison: baseline ===
Saved condition outputs:
  reused original panel cache: True
  whole-review master: /Users/eveyhuang/Documents/NICO/human-AI-proposal/results/tables/baseline/reviews/rephrased/shared_cache/review_diversity_proposal_master.csv
  strengths master: /Users/eveyhuang/Documents/NICO/human-AI-proposal/results/tables/baseline/reviews/rephrased/shared_cache/review_diversity_strengths_proposal_master.csv
  weakness master: /Users/eveyhuang/Documents/NICO/human-AI-proposal/results/tables/baseline/reviews/rephrased/shared_cache/review_diversity_weakness_proposal_master.csv

=== 6b rephrased review comparison: one_at_a_time ===
Saved condition outputs:
  reused original panel cache: True
  whole-review master: /Users/eveyhuang/Documents/NICO/human-AI-proposal/results/tables/one_at_a_time/reviews/rephrased/shared_cache/review_diversity_proposal_master.csv
  strengths master: /Users/eveyhuang/Documents/NICO/human-AI-proposal/results/tables/one_at_a_ti

In [8]:
cross_tables_dir = PROJECT_ROOT / 'results' / 'tables' / 'reviews' / TEXT_VERSION / 'cross_condition'
cross_figures_dir = PROJECT_ROOT / 'results' / 'figures' / 'reviews' / TEXT_VERSION / 'cross_condition'
cross_tables_dir.mkdir(parents=True, exist_ok=True)
cross_figures_dir.mkdir(parents=True, exist_ok=True)

cross_condition_whole_tests_df = cross_condition_summary_table(cross_condition_whole_test_frames)
cross_condition_whole_proposal_df = pd.concat(cross_condition_whole_proposal_frames, ignore_index=True)
cross_condition_field_tests_df = cross_condition_summary_table(cross_condition_field_test_frames)
condition_contrast_df = cross_condition_ai_contrast_table(cross_condition_whole_proposal_df)

cross_condition_whole_tests_df.to_csv(cross_tables_dir / 'review_diversity_cross_condition_summary.csv', index=False)
condition_contrast_df.to_csv(cross_tables_dir / 'review_diversity_condition_contrasts.csv', index=False)
cross_condition_field_tests_df.to_csv(cross_tables_dir / 'review_diversity_field_specific_cross_condition_summary.csv', index=False)


cross_facet_tests_df = pd.concat(cross_condition_facet_test_frames, ignore_index=True) if cross_condition_facet_test_frames else pd.DataFrame()
cross_facet_gradient_df = pd.concat(cross_condition_facet_gradient_frames, ignore_index=True) if cross_condition_facet_gradient_frames else pd.DataFrame()
cross_facet_paired_df = pd.concat(cross_condition_facet_paired_frames, ignore_index=True) if cross_condition_facet_paired_frames else pd.DataFrame()
cross_facet_tests_df.to_csv(cross_tables_dir / 'facet_diversity_tests.csv', index=False)
cross_facet_gradient_df.to_csv(cross_tables_dir / 'facet_diversity_gradient.csv', index=False)
cross_facet_paired_df.to_csv(cross_tables_dir / 'facet_review_paired_long.csv', index=False)
if cross_condition_facet_curve_frames:
    cross_facet_curves_df = pd.concat(cross_condition_facet_curve_frames, ignore_index=True)
    try:
        cross_facet_curves_df.to_parquet(cross_tables_dir / 'facet_diversity_curves.parquet', index=False)
    except Exception:
        cross_facet_curves_df.to_csv(cross_tables_dir / 'facet_diversity_curves.csv', index=False)

confirmatory_df = cross_condition_whole_tests_df[(cross_condition_whole_tests_df['subset_label'] == 'all_proposals') & (cross_condition_whole_tests_df['metric_class'] == 'confirmatory')].copy()

fig, axes = plt.subplots(1, len(CONFIRMATORY_METRICS), figsize=(6 * len(CONFIRMATORY_METRICS), 5), sharey=False)
if len(CONFIRMATORY_METRICS) == 1:
    axes = [axes]
for ax, metric in zip(axes, CONFIRMATORY_METRICS):
    sub = confirmatory_df[confirmatory_df['metric'] == metric].copy()
    sns.barplot(data=sub, x='condition', y='effect_human_minus_ai_mean', hue='comparison_label', ax=ax)
    ax.axhline(0.0, color='black', linewidth=1.0)
    ax.set_title(metric)
    ax.set_xlabel('')
    ax.set_ylabel('Mean paired difference (Human - AI)')
fig.suptitle('Cross-condition review-diversity effects (rephrased confirmatory metrics)')
fig.tight_layout()
fig.savefig(cross_figures_dir / 'review_diversity_effects_confirmatory_cross_condition.png', dpi=200, bbox_inches='tight')
plt.close(fig)

fig, axes = plt.subplots(1, len(CONFIRMATORY_METRICS), figsize=(6 * len(CONFIRMATORY_METRICS), 5), sharey=False)
if len(CONFIRMATORY_METRICS) == 1:
    axes = [axes]
for ax, metric in zip(axes, CONFIRMATORY_METRICS):
    sub = confirmatory_df[confirmatory_df['metric'] == metric].copy()
    sns.barplot(data=sub, x='condition', y='ai_to_human_ratio_mean', hue='comparison_label', ax=ax)
    ax.axhline(1.0, color='black', linewidth=1.0, linestyle='--')
    ax.set_title(metric)
    ax.set_xlabel('')
    ax.set_ylabel('AI / Human diversity-retained ratio')
fig.suptitle('Cross-condition retained-diversity ratios (rephrased confirmatory metrics)')
fig.tight_layout()
fig.savefig(cross_figures_dir / 'review_diversity_retained_ratio_confirmatory_cross_condition.png', dpi=200, bbox_inches='tight')
plt.close(fig)

field_confirmatory_df = cross_condition_field_tests_df[(cross_condition_field_tests_df['subset_label'] == 'all_proposals') & (cross_condition_field_tests_df['metric_class'] == 'confirmatory')].copy()
fig, axes = plt.subplots(1, len(CONFIRMATORY_METRICS), figsize=(6 * len(CONFIRMATORY_METRICS), 5), sharey=False)
if len(CONFIRMATORY_METRICS) == 1:
    axes = [axes]
for ax, metric in zip(axes, CONFIRMATORY_METRICS):
    sub = field_confirmatory_df[field_confirmatory_df['metric'] == metric].copy()
    sns.barplot(data=sub, x='condition', y='effect_human_minus_ai_mean', hue='field_branch', ax=ax)
    ax.axhline(0.0, color='black', linewidth=1.0)
    ax.set_title(metric)
    ax.set_xlabel('')
    ax.set_ylabel('Mean paired difference (Human - AI)')
fig.suptitle('Cross-condition field-specific review-diversity effects')
fig.tight_layout()
fig.savefig(cross_figures_dir / 'review_diversity_field_specific_effects_cross_condition.png', dpi=200, bbox_inches='tight')
plt.close(fig)

display(cross_condition_whole_tests_df.sort_values(['condition', 'comparison', 'metric', 'subset_label']).head(40))
display(cross_condition_field_tests_df.sort_values(['field_branch', 'condition', 'comparison', 'metric', 'subset_label']).head(40))
display(condition_contrast_df.sort_values(['comparison', 'metric', 'left_condition', 'right_condition']).head(40))


,condition,comparison,comparison_label,metric,metric_class,subset_label,n_proposals,human_metric_mean,ai_metric_mean,effect_human_minus_ai_mean,effect_human_minus_ai_median,ai_to_human_ratio_mean,wilcoxon_statistic,p_value,paired_rank_biserial,field_branch
0,baseline,all_ai,All AI,centroid_loo,exploratory,all_proposals,23,0.031170,0.042641,-0.011471,-0.012949,1.368027,6.0,3.337860e-06,-0.739130,whole_review
36,baseline,all_ai,All AI,centroid_loo,exploratory,human_n_gte_3,22,0.030540,0.041682,-0.011142,-0.012512,1.364842,6.0,6.675720e-06,-0.727273,whole_review
1,baseline,all_ai,All AI,global_centroid_dist,exploratory,all_proposals,23,0.015839,0.021728,-0.005890,-0.006317,1.371847,6.0,3.337860e-06,-0.739130,whole_review
37,baseline,all_ai,All AI,global_centroid_dist,exploratory,human_n_gte_3,22,0.016044,0.021985,-0.005941,-0.006683,1.370295,6.0,6.675720e-06,-0.727273,whole_review
2,baseline,all_ai,All AI,mean_pairwise,confirmatory,all_proposals,23,0.043840,0.059855,-0.016015,-0.018714,1.365319,6.0,3.337860e-06,-0.739130,whole_review
38,baseline,all_ai,All AI,mean_pairwise,confirmatory,human_n_gte_3,22,0.043785,0.059678,-0.015893,-0.017714,1.362972,6.0,6.675720e-06,-0.727273,whole_review
3,baseline,all_ai,All AI,medoid_dist,exploratory,all_proposals,23,0.027869,0.037846,-0.009977,-0.010169,1.358001,2.0,7.152557e-07,-0.913043,whole_review
39,baseline,all_ai,All AI,medoid_dist,exploratory,human_n_gte_3,22,0.028112,0.038117,-0.010005,-0.011087,1.355907,2.0,1.430511e-06,-0.909091,whole_review
4,baseline,all_ai,All AI,mst_dispersion,exploratory,all_proposals,23,0.038680,0.051181,-0.012502,-0.013898,1.323211,4.0,1.668930e-06,-0.826087,whole_review
40,baseline,all_ai,All AI,mst_dispersion,exploratory,human_n_gte_3,22,0.038391,0.050610,-0.012219,-0.013756,1.318288,4.0,3.337860e-06,-0.818182,whole_review


,condition,comparison,comparison_label,metric,metric_class,subset_label,n_proposals,human_metric_mean,ai_metric_mean,effect_human_minus_ai_mean,effect_human_minus_ai_median,ai_to_human_ratio_mean,wilcoxon_statistic,p_value,paired_rank_biserial,field_branch
0,baseline,all_ai,All AI,centroid_loo,exploratory,all_proposals,23,0.048673,0.060758,-0.012085,-0.008651,1.248283,19.0,0.000073,-0.826087,strengths
36,baseline,all_ai,All AI,centroid_loo,exploratory,human_n_gte_3,22,0.048450,0.059338,-0.010888,-0.007718,1.224728,19.0,0.000146,-0.818182,strengths
1,baseline,all_ai,All AI,global_centroid_dist,exploratory,all_proposals,23,0.025116,0.030938,-0.005823,-0.004817,1.231827,22.0,0.000128,-0.826087,strengths
37,baseline,all_ai,All AI,global_centroid_dist,exploratory,human_n_gte_3,22,0.025644,0.031285,-0.005641,-0.003915,1.219960,22.0,0.000256,-0.818182,strengths
2,baseline,all_ai,All AI,mean_pairwise,confirmatory,all_proposals,23,0.068651,0.084799,-0.016148,-0.012434,1.235220,22.0,0.000128,-0.826087,strengths
38,baseline,all_ai,All AI,mean_pairwise,confirmatory,human_n_gte_3,22,0.069336,0.084472,-0.015136,-0.010591,1.218302,22.0,0.000256,-0.818182,strengths
3,baseline,all_ai,All AI,medoid_dist,exploratory,all_proposals,23,0.044176,0.053850,-0.009674,-0.010003,1.218995,19.0,0.000073,-0.739130,strengths
39,baseline,all_ai,All AI,medoid_dist,exploratory,human_n_gte_3,22,0.044966,0.054207,-0.009241,-0.008385,1.205512,19.0,0.000146,-0.727273,strengths
4,baseline,all_ai,All AI,mst_dispersion,exploratory,all_proposals,23,0.059609,0.073134,-0.013525,-0.010848,1.226899,19.0,0.000073,-0.739130,strengths
40,baseline,all_ai,All AI,mst_dispersion,exploratory,human_n_gte_3,22,0.059883,0.072277,-0.012394,-0.010784,1.206972,19.0,0.000146,-0.727273,strengths


,comparison,comparison_label,metric,left_condition,right_condition,n_proposals,left_mean_ai_metric,right_mean_ai_metric,mean_difference_left_minus_right,median_difference_left_minus_right,wilcoxon_statistic,p_value
0,all_ai,All AI,centroid_loo,baseline,one_at_a_time,23,0.042641,0.023455,0.019186,0.018752,0.0,2.384186e-07
1,all_ai,All AI,centroid_loo,baseline,persona,23,0.042641,0.023449,0.019192,0.018695,0.0,2.384186e-07
2,all_ai,All AI,centroid_loo,one_at_a_time,persona,23,0.023455,0.023449,0.000006,0.000034,136.0,9.643126e-01
3,all_ai,All AI,global_centroid_dist,baseline,one_at_a_time,23,0.021728,0.012033,0.009696,0.010260,0.0,2.384186e-07
4,all_ai,All AI,global_centroid_dist,baseline,persona,23,0.021728,0.012034,0.009694,0.009625,0.0,2.384186e-07
5,all_ai,All AI,global_centroid_dist,one_at_a_time,persona,23,0.012033,0.012034,-0.000001,0.000019,129.0,7.997725e-01
6,all_ai,All AI,mean_pairwise,baseline,one_at_a_time,23,0.059855,0.033222,0.026633,0.027121,0.0,2.384186e-07
7,all_ai,All AI,mean_pairwise,baseline,persona,23,0.059855,0.033215,0.026640,0.026898,0.0,2.384186e-07
8,all_ai,All AI,mean_pairwise,one_at_a_time,persona,23,0.033222,0.033215,0.000007,0.000050,134.0,9.168475e-01
9,all_ai,All AI,medoid_dist,baseline,one_at_a_time,23,0.037846,0.021328,0.016518,0.016790,0.0,2.384186e-07
